# Task 2.4: readout design

This notebook only reads `results/` and `figures/`. Set `RERUN = True` to recompute the linear-response frontier. The two-mode check is a separate command and was not run in the lane that wrote these records.

**Why the shaped pulse can beat a square, in linear response.** About a well, the memory-buffer pair is one damped resonator. Its transfer function peaks at the ringing pole (about 3.14 MHz, Q about 2.5), so a drive that puts its energy there moves the buffer more per unit of ∫ε² dt than a square, which spends amplitude on other frequencies and on the sharp edges that ring. Matched cost means the two pulses are scaled to the same ∫ε² dt, the same ∫|δa|² dt, or the same quasi-static bit-flip budget before their SNRs are compared. The singular vector is optimal only inside that linear map. It is not, by itself, a two-mode win.

In [ ]:
import os, json
from pathlib import Path
if not os.path.isdir('src'):
    os.chdir('..')
from IPython.display import Image, display, Markdown
RERUN = False
if RERUN:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'tasks.task_2_4c_pulses'])

def show(name, w=900):
    path = Path('figures') / f'{name}.png'
    if path.exists():
        display(Image(path.as_posix(), width=w))
    else:
        display(Markdown(f'*missing `{path}`*'))

def records(prefix):
    root = Path('results/records')
    if not root.is_dir():
        return []
    out = []
    for p in sorted(root.glob(prefix + '*.json')):
        out.append(json.loads(p.read_text()))
    return out


## 2.4a summary

Circuit-level g₂ and the parasitic terms live in `docs/derivation_2_4a.md` (lane l5) and in `t2_4a.*` records. This notebook does not re-derive them.

In [ ]:
from pathlib import Path
print('derivation present:', Path('docs/derivation_2_4a.md').exists())
recs = records('t2_4a.')
print(f'{len(recs)} t2_4a records')
for r in recs[:8]:
    print(f"{r['id']}: {r['value']} {r['units']}")


## 2.4b trade-off

The quasi-static Γ_X(ε) grid is `results/arrays/t2_4b.grid.npz`. If it is absent, 2.4c uses the analytic Dubovitskii proxy and says so in the cost-record notes.

In [ ]:
from pathlib import Path
grid = Path('results/arrays/t2_4b.grid.npz')
print('l4b grid present:', grid.exists())
for r in records('t2_4b.')[:8]:
    print(f"{r['id']}: {r['value']} {r['units']}")


## 2.4c candidates and the linear-response frontier

Square, ramps, CLEAR, the top singular vector of R_b (SVD), and the generalized pulse against leakage. The SNR records are at drive energy matched to a square of amplitude 0.1 g₂. Because the analytic C_X is proportional to ∫ε² dt, that is also matched cost (iii) until the l4b grid lands.

In [ ]:
show('task_2_4c_pulses')
show('task_2_4c_comparison')
rows = [r for r in records('t2_4c.snr.linresp.') if r['params'].get('T_us') == 1.0 and abs(r['params'].get('alpha', 0) - 2.1) < 1e-9]
print(f"{'pulse':22} {'SNR':12} {'gain_E':8} {'gain_L':8} {'gain_C':8}")
for r in rows:
    n = r.get('numerics') or {}
    print(f"{r['params'].get('pulse'):22} {r['value']:12.6f} {n.get('gain_energy', float('nan')):8.4f} "
          f"{n.get('gain_leakage', float('nan')):8.4f} {n.get('gain_cost', float('nan')):8.4f}")


## 2.4d validation

A two-mode SNR within 10% of linear response, at a converged truncation, is required before any pulse is called a win on the full model. If `t2_4c.snr.td.*` records are missing, that check has not been run. Do not read a linear-response gain as a two-mode result.

The direct bit-flip train at α = 1.0 (square and the energy-matched winner, 40 µs, plus a tighter tolerance repeat) is a queued job, not a number in these records.

In [ ]:
show('task_2_4c_validation')
td = records('t2_4c.snr.td.')
print(f'{len(td)} two-mode SNR records')
for r in td:
    print(r['id'], r['value'], r.get('cross_check'))
